In [ ]:
import numpy as np
from numba import cuda
import math

BLOCK_SIZE = 512

## Per-block exclusive scan (Blelloch).
# arr:          (n,) int32 input
# scan:         (n,) int32 output, exclusive scan within each block
# block_totals: (num_blocks,) int32 output, the full sum of each block
@cuda.jit
def exclusive_scan_kernel(arr, scan, block_totals):
  t = cuda.threadIdx.x
  b = cuda.blockIdx.x
  i = cuda.blockIdx.x * BLOCK_SIZE + cuda.threadIdx.x

  # allocate shared memory for intermediate calculations
  partial_scan = cuda.shared.array(BLOCK_SIZE, dtype=np.int32)
  if i < arr.size:
    partial_scan[t] = arr[i]
  else:
    partial_scan[t] = 0

  # up-sweep
  stride = 1
  while stride < BLOCK_SIZE: # goes up until 256
    cuda.syncthreads()
    if (t+1) % (2*stride) == 0:
      partial_scan[t] = partial_scan[t] + partial_scan[t-stride]
    stride *= 2

  # the root now holds the block total: save it, then set it to 0
  # (same thread wrote the root in the last up-sweep step, so no sync needed)
  if t == BLOCK_SIZE-1:
    block_totals[b] = partial_scan[t]
    partial_scan[t] = 0

  # down-sweep
  stride = BLOCK_SIZE // 2 # starts from 256
  while stride >= 1:
    cuda.syncthreads()
    if (t+1) % (2*stride) == 0:
      temp = partial_scan[t]
      partial_scan[t] = partial_scan[t] + partial_scan[t-stride]
      partial_scan[t-stride] = temp
    stride //= 2

  # wait for the last down-sweep step (thread t+1 writes partial_scan[t])
  cuda.syncthreads()

  if i < scan.size:
    scan[i] = partial_scan[t]

@cuda.jit
def add_offset_kernel(split_scan, block_offsets, full_scan):
  b = cuda.blockIdx.x
  i = cuda.blockIdx.x * BLOCK_SIZE + cuda.threadIdx.x

  if i < full_scan.size:
    full_scan[i] = split_scan[i] + block_offsets[b]


## Recursive exclusive scan of an arbitrary-length device array.
# d_arr: (n,) int32 device array
# returns (scan, total): scan is (n,) int32, total is a (1,) int32 device array
def exclusive_scan(d_arr):
  n = d_arr.size
  num_blocks = math.ceil(n / BLOCK_SIZE)

  # compute independent exclusive scan of each block, plus each block's total
  split_scan = cuda.device_array(n, dtype=np.int32)
  block_totals = cuda.device_array(num_blocks, dtype=np.int32)
  exclusive_scan_kernel[num_blocks, BLOCK_SIZE](d_arr, split_scan, block_totals)

  # base case: one block, so its scan is already global and its total is the full sum
  if num_blocks == 1:
    return split_scan, block_totals

  # block offsets are the exclusive scan of block totals (recurse: may itself be > 1 block)
  block_offsets, total = exclusive_scan(block_totals)

  # add offsets to each block
  full_scan = cuda.device_array(n, dtype=np.int32)
  add_offset_kernel[num_blocks, BLOCK_SIZE](split_scan, block_offsets, full_scan)

  return full_scan, total


## inputs
# flattened_mask: (B*L,)

## outputs
# pos: (B*L,), the result of flattened_mask after an exclusive scan
# N_total: (1,) device array, the sum of flattened_mask
def generate_pos(flattened_mask):
  if isinstance(flattened_mask, np.ndarray):
    d_flattened_mask = cuda.to_device(np.ascontiguousarray(flattened_mask, dtype=np.int32))
  else:
    d_flattened_mask = flattened_mask  # already an int32 device array
  return exclusive_scan(d_flattened_mask)


## Scatter each valid token's row into its slot in packed.
# Block is (32, TOKENS_PER_BLOCK): threadIdx.y picks the token, threadIdx.x strides over D,
# so adjacent threads touch adjacent features (coalesced reads and writes).
TOKENS_PER_BLOCK = 8

@cuda.jit
def scatter_kernel(x, flattened_mask, pos, packed):
  L = x.shape[1]
  D = x.shape[2]
  tok = cuda.blockIdx.x * TOKENS_PER_BLOCK + cuda.threadIdx.y

  if tok < flattened_mask.size and flattened_mask[tok] == 1:
    b = tok // L
    l = tok % L
    row = pos[tok]
    d = cuda.threadIdx.x
    while d < D:
      packed[row, d] = x[b, l, d]
      d += cuda.blockDim.x

## cu_seqlens[b] = pos[b*L] (number of valid tokens before sequence b), cu_seqlens[B] = N_total
@cuda.jit
def cu_seqlens_kernel(pos, N_total, L, cu_seqlens):
  b = cuda.grid(1)
  B = cu_seqlens.size - 1
  if b < B:
    cu_seqlens[b] = pos[b * L]
  elif b == B:
    cu_seqlens[B] = N_total[0]

## inputs
# x: (B, L, D), padded batch of B sequences, each padded to length L
# mask: (B, L), 1 for a real (valid) token, 0 for padding

## outputs
# packed: (N_total, D), the valid tokens of all sequences concatenated in order
#         (sequence 0's valid tokens, then sequence 1's, ...), where N_total = mask.sum()
# cu_seqlens: (B + 1,), cumulative sequence lengths: sequence b occupies
#             packed[cu_seqlens[b] : cu_seqlens[b+1]], with cu_seqlens[0] = 0 and cu_seqlens[B] = N_total
def build_packed(x, mask):
  B, L, D = x.shape

  d_x = cuda.to_device(np.ascontiguousarray(x)) if isinstance(x, np.ndarray) else x
  d_mask = cuda.to_device(np.ascontiguousarray(mask, dtype=np.int32).reshape(-1))
  pos, N_total = generate_pos(d_mask)

  # output size depends on the data, so copy N_total back to the host (sync point)
  n_total = int(N_total.copy_to_host()[0])

  packed = cuda.device_array((n_total, D), dtype=d_x.dtype)
  cu_seqlens = cuda.device_array(B+1, dtype=np.int32)

  # scatter valid tokens into packed
  num_tokens = B * L
  scatter_blocks = math.ceil(num_tokens / TOKENS_PER_BLOCK)
  scatter_kernel[scatter_blocks, (32, TOKENS_PER_BLOCK)](d_x, d_mask, pos, packed)

  # sequence boundaries come straight from the scan
  threads = 128
  cu_seqlens_kernel[math.ceil((B + 1) / threads), threads](pos, N_total, L, cu_seqlens)

  return packed, cu_seqlens
